# Importing a public dataset : DeepLabCut pose tracking

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

An entry point to **larvaworld** built around a dataset produced with
[DeepLabCut](https://www.mackenziemathislab.org/deeplabcut), the pose-estimation toolbox.
It shows that larvaworld reads DeepLabCut exports directly, and - more importantly - what
you have to know about such an export before the numbers mean anything.

**The dataset.** DeepLabCut results from Greaney, Heckscher and Kaufman (2025), who studied
how movement is coordinated along the larval body. Larvae were filmed at high magnification
while crawling, and a network was trained to place **26 landmarks** along the body: a left
and a right point for the head, three thoracic and eight abdominal segments, and the tail.
larvaworld averages each left/right pair into one midline point, giving a **13-point
midline**.

**The two groups.** The archive holds three folders. `SideView` is a lateral view and is
**ignored** here. The two top-down sets, recorded a year apart, are imported as **two
separate datasets** and kept separate throughout, so every figure shows both.

**What this notebook is really about.** A DeepLabCut export is just landmark coordinates in
**pixels**. It carries no arena, no frame rate and no scale. Section 2 shows what happens
when you guess the scale wrong, and how larvaworld catches it.

This notebook is one of a series; a blank version is available as
`import_public_dataset_template.ipynb`.

## Setup

Importing larvaworld initializes its configuration registry : some components are loaded
from disc and the rest are built on the fly. `VERBOSE = 1` makes the import report what it
is doing, which is worth watching the first time.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython.display import display

import larvaworld
from larvaworld.lib import reg, sim, util
from larvaworld.lib.reg.generators import ReplayConf

larvaworld.VERBOSE = 1

# The name of this experiment. It labels the imported datasets and the output folders.
EXPERIMENT_NAME = "DeepLabCut"

MEDIA_DIR = Path(f"./media/{EXPERIMENT_NAME}")
plot_dir = (MEDIA_DIR / "plots").as_posix()
video_dir = (MEDIA_DIR / "videos").as_posix()

# Rendering the replay videos needs ffmpeg and takes several minutes.
MAKE_VIDEOS = False

ds = []  # the imported datasets, filled in further below

import shutil
import zipfile

import requests


def fetch_dataset(expect, archive, url=None, extract=True):
    """Make a dataset available locally, doing as little work as possible.

    Resolves in three steps, reporting which one it took :
      1. the extracted data is already there  -> nothing happens
      2. the archive is there but not unpacked -> unpack only
      3. neither                               -> download, then unpack

    Args:
        expect: path that exists once the data is unpacked.
        archive: path of the downloaded archive.
        url: where to download the archive from, if it is missing.
        extract: whether the archive can be unpacked here. RAR archives cannot.

    Returns:
        True if `expect` is available afterwards.
    """
    expect, archive = Path(expect), Path(archive)
    if expect.exists():
        print(f"[1/3] already present, nothing to do : {expect}")
        return True

    if not archive.exists():
        if url is None:
            print(f"[3/3] missing and no download link given : {archive}")
            return False
        archive.parent.mkdir(parents=True, exist_ok=True)
        print(f"[3/3] downloading {url}\n      -> {archive}")
        tmp = archive.with_suffix(archive.suffix + ".part")
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            done = 0
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    done += len(chunk)
                    if total:
                        print(
                            f"      {done / 1e6:8.0f} / {total / 1e6:.0f} MB", end="\r"
                        )
        tmp.rename(archive)
        print(f"\n      downloaded {archive.stat().st_size / 1e6:.0f} MB")
    else:
        print(f"[2/3] archive already downloaded : {archive}")

    if not extract:
        print(f"      this archive cannot be unpacked from Python. Extract it with")
        print(f"      7-Zip, WinRAR or unrar so that this exists :\n      {expect}")
        return expect.exists()

    print(f"      unpacking -> {expect.parent}")
    with zipfile.ZipFile(archive) as z:
        z.extractall(expect.parent)
    return expect.exists()

## Section 1 : Get the data

The dataset is openly available on figshare.

- **Record** : <https://figshare.com/articles/dataset/DeepLabCut_results/31508029>
- **Archive** : <https://figshare.com/ndownloader/articles/31508029/versions/1>

Nothing is downloaded or unpacked if it is already on disc.

In [ ]:
DOWNLOAD_ROOT = Path.home() / "Downloads" / "31508029"
URL = "https://figshare.com/ndownloader/articles/31508029/versions/1"

fetch_dataset(
    expect=DOWNLOAD_ROOT / "DLC_files",
    archive=DOWNLOAD_ROOT.with_suffix(".zip"),
    url=URL,
)

RAW_FOLDER = (DOWNLOAD_ROOT / "DLC_files").as_posix()

# The two top-down sets. The SideView folder is a lateral view and is not used here.
GROUPS = {"TopDown-2023-07-05": "black", "TopDown-2024-02-17": "red"}

DATA_AVAILABLE = all((Path(RAW_FOLDER) / g).is_dir() for g in GROUPS)
if DATA_AVAILABLE:
    for g in GROUPS:
        print(f"{g:22s} : {len(list((Path(RAW_FOLDER) / g).glob('*.csv')))} recordings")

## Section 2 : Import to larvaworld

Three things have to be specified : **where the data is**, **which tracker wrote it**, and
**which tracks to keep**. For DeepLabCut a fourth is unavoidable, and it is the interesting
one : **how big a pixel is**.

### What the tracker recorded, and what it did not

Before importing, it is worth knowing which properties of a recording are written down
somewhere and which are not. For most published tracking data the picture is this :

| property | stated in the archive? | larvaworld can derive it |
|---|---|---|
| recording duration | usually, in the tracker's settings | not needed |
| stimulus protocol | usually, in the tracker's settings | not needed |
| **frame rate** | **often not** | **yes**, from the timestamps |
| **number of midline points** | **no** | **yes**, from the coordinates |
| **arena dimensions** | **no** | partly, see below |
| **pixel-to-millimetre scale** | **no** | no - you have to know it |

The highlighted rows are the ones that matter for the import, and they are the ones least
likely to be recorded. larvaworld therefore derives what it can from the data itself :

- **Frame rate.** Many trackers record at a variable rate, so a single nominal frame rate
  does not describe them. When a lab format declares a variable framerate, the timestep is
  measured from the timestamps and used for the whole import, including the stored dataset.
  Pass `estimate_dt=False` to keep the nominal value. Formats whose data carries no
  timestamps at all cannot use this, and need the frame rate set by hand.
- **Midline points.** Counted from the data and used whenever it disagrees with the
  expected number. Pass `estimate_midline_points=False` to switch this off.
- **Arena dimensions.** Estimated from the area the animals actually covered, which makes
  it a *lower bound* : larvae that never reach the rim make the arena look smaller than it
  is. It is therefore **off by default**. Pass `estimate_arena_dimensions=True`.

**The spatial scale is the one thing you must bring yourself.** If a tracker exports pixels
rather than millimetres, nothing in the coordinates reveals the conversion factor. A quick
sanity check settles which case you are in : the summed length of a larva's midline should
be a few millimetres for a third-instar larva. larvaworld applies that same check on
import and refuses data implying an impossible animal.

### Two things this export does not tell us

**The frame rate.** DeepLabCut writes one row per video frame and no timestamps at all, so
there is nothing for larvaworld to measure - `estimate_dt` cannot help here. The rate has to
come from outside the data. In this dataset the file names carry it: every top-down
recording ends in `20fps`, so the frame rate is set to 20 Hz by hand.

**The scale.** The coordinates are in pixels. The repository does not report how many
pixels make a millimetre, and larvaworld works in millimetres throughout - lengths,
velocities, dispersal and the arena are all metric. A scale therefore has to be assumed, and
the choice is not cosmetic: get it wrong and every distance in the analysis is wrong by the
same factor.

Let us start by not saying anything about it, which is the state a DeepLabCut export
arrives in.

In [ ]:
from larvaworld.lib.process.import_aux import DLCScaleValidationError

lf = reg.conf.LabFormat.get("DeepLabCut")
lf.tracker.fr = 20  # from the file names, not from the data
# Note what is *not* set here: filesystem.pixel_to_mm is left as it comes, unspecified.

constraints = util.AttrDict({"min_duration_in_sec": 20})
enr_kws = util.AttrDict(
    {
        "proc_keys": ["angular", "spatial"],
        "anot_keys": ["bout_detection"],
        "traj2origin": True,
        "tor_durs": [20],
        "dsp_starts": [0],
        "dsp_stops": [40, 60],
    }
)

if DATA_AVAILABLE:
    try:
        lf.import_dataset(
            raw_folder=RAW_FOLDER,
            parent_dir="TopDown-2023-07-05",
            id="wrong_scale",
            color="black",
            save_dataset=False,
            enrich_conf=enr_kws,
            **constraints,
        )
    except DLCScaleValidationError as e:
        print(f"refused, and rightly so :\n\n  {e}")

The import refuses to proceed. With no conversion given, larvaworld has no choice but to read
the coordinates as if they were already millimetres, and so believes it has been handed
animals **a few hundred millimetres long** - about the length of a forearm. A third-instar
*Drosophila* larva is a few millimetres, so the numbers cannot be millimetres.

That check is worth dwelling on, because it is the only thing standing between a wrong
constant and a plausible-looking analysis. Nothing else downstream would have complained:
the trajectories would have been drawn, the dispersal curves would have risen, and every
number would have been ten times too large.

That refusal is also the measurement we need. The tracked midline is about 390 units long,
and those units are pixels; a real larva of about 3.9 mm therefore implies **100 pixels per
millimetre**. We adopt that, and say plainly what it is - **an assumption**, made because
the repository does not report the true value.

In [ ]:
lf.filesystem.pixel_to_mm = 0.01  # 100 px per mm - assumed, not reported

refIDs = [f"{EXPERIMENT_NAME}.{g}" for g in GROUPS]

if DATA_AVAILABLE:
    ds = [
        lf.import_dataset(
            raw_folder=RAW_FOLDER,
            parent_dir=g,
            id=g,
            refID=refID,
            group_id=EXPERIMENT_NAME,
            color=color,
            save_dataset=True,
            # The arena is unknown for this dataset, so let it be measured.
            estimate_arena_dimensions=True,
            enrich_conf=enr_kws,
            **constraints,
        )
        for (g, color), refID in zip(GROUPS.items(), refIDs)
    ]
    for d in ds:
        dims = tuple(round(v * 1000, 1) for v in d.config.env_params.arena.dims)
        print(
            f"{d.id:22s} : {d.config.N} larvae, {d.config.Npoints} midline points, "
            f"body {d.e['length'].mean() * 1000:.2f} mm, field {dims} mm"
        )

The larvae now come out at a believable length, which is the whole of the evidence for the
assumed scale - it is self-consistent, not independently verified.

Note the measured field of view: a few millimetres across. These are **high-magnification
recordings of one crawling animal**, not an arena being explored. The dispersal figures
below are therefore about how far a larva travelled through a small window, which is a
different question from the one the same plots answer for an open-field dataset. Read them
with that in mind.

### Reloading in a later session

In [ ]:
if not ds:
    if all(refID in reg.conf.Ref.confIDs for refID in refIDs):
        ds = [reg.loadRef(id=refID, load=True) for refID in refIDs]
        print("Loaded :", [d.id for d in ds])
    else:
        print("These datasets have not been imported yet. Run Section 2 first.")

## Section 3 : Data analysis and plotting

larvaworld ships a library of plotting routines, each registered under a short name. You
pick one by name and hand it the datasets you want compared - the group colors and labels
are taken from the datasets themselves, so every figure is consistent.

In [ ]:
# The available plots, by their unique IDs
print(reg.graphs.ks)

In [ ]:
# Arguments shared by every plot below. Figures are also written to `plot_dir`.
plot_kws = {"datasets": ds, "save_to": plot_dir, "show": False, "subfolder": None}

### The trajectories

First, simply what the larvae did : their paths over the analysed window.

In [ ]:
if ds:
    display(reg.graphs.run("trajectories", **plot_kws))

The same trajectories, but each one translated so that it starts at the origin, and colored
by group. This removes the arbitrary starting position of each animal and makes the *shape
and extent* of the paths directly comparable.

In [ ]:
if ds:
    display(
        reg.graphs.run("trajectories", mode="origin", single_color=True, **plot_kws)
    )

### Endpoint metrics

A boxplot of endpoint metrics - one value per larva, summarising its whole track. Each plot
routine has a default selection, but you can always name the metrics you want by their
short keys, as done here.

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=[
                "l",
                "fsv",
                "sv_mu",
                "run_tr",
                "pau_tr",
                "tor20_mu",
                "dsp_0_40_fin",
                "b_mu",
                "bv_mu",
            ],
            **plot_kws,
        )
    )

And a composite figure summarising exploration behavior across the groups.

In [ ]:
if ds:
    display(reg.graphs.run("exploration summary", **plot_kws))

### Dispersal

Dispersal is the distance of a larva from where it started. We compare the two recording sets on it
in three increasingly informative ways.

**1. As an endpoint statistic.** The mean, final and maximum dispersal reached during the
analysed window - one number per larva, summarised as a boxplot per group.

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=["dsp_0_60_mu", "dsp_0_60_fin", "dsp_0_60_max"],
            **plot_kws,
        )
    )

**2. As a timecourse.** Dispersal plotted against time, showing both the mean and the
variance of each group : not just how far the groups got, but how fast and how consistently.

In [ ]:
if ds:
    display(reg.graphs.run("dispersal", **plot_kws))

In [ ]:
if ds:
    display(reg.graphs.run("dispersal", range=(0, 60), **plot_kws))

**3. Alongside the paths that produced it.** The summary versions place the timecourse next
to the corresponding trajectories, which makes the link between curve and behavior
immediate.

In [ ]:
if ds:
    display(reg.graphs.run("dispersal summary", **plot_kws))

## Section 4 : Visualize the dataset

A *replay* is a simulation whose agents are driven by recorded data instead of a model. It
gives you the same visualization tools you would use on a simulation - here, the
trajectories of all larvae of a group, transposed to a common origin and drawn as
accumulating trails. Both datasets are rendered and then placed side by side.

Rendering needs `ffmpeg` (installed with larvaworld via `imageio_ffmpeg`) and takes a few
minutes per group, so it is off by default. Set `MAKE_VIDEOS = True` in the Setup cell.

In [ ]:
def run_replay(d):
    """Render one dataset's tracks to a video file in `video_dir`."""
    screen_kws = {
        "vis_mode": "video",
        "show_display": False,
        "draw_contour": False,
        "draw_midline": False,
        "draw_centroid": False,
        "visible_trails": True,
        "save_video": True,
        "fps": 1,
        "video_file": d.id,
        "media_dir": video_dir,
    }
    replay_conf = ReplayConf(
        transposition="origin", time_range=(0, 60), track_point=d.c.point_idx
    ).nestedConf
    rep = sim.ReplayRun(
        dataset=d,
        parameters=replay_conf,
        id=f"{d.id}_replay",
        screen_kws=screen_kws,
    )
    return rep.run()

In [ ]:
if MAKE_VIDEOS and ds:
    for d in ds:
        run_replay(d)

Finally the videos are stacked side by side into a single one, giving a direct visual
comparison of the groups.

In [ ]:
if MAKE_VIDEOS and ds:
    from larvaworld.lib.util.combining import combine_videos

    combine_videos(file_dir=video_dir, save_as="combined.mp4")
    print(f"Written to {video_dir}/combined.mp4")

## A few words on the lab format

Every tracker writes its own files, so larvaworld reads each one through a named **lab
format**. A lab format knows how a lab's files are laid out and how their contents must be
preprocessed, which is why the import above needed nothing more than a folder and a name.

| lab format | suits data that looks like |
|---|---|
| `Jovanic` | one file per recorded quantity, all animals stacked together |
| `Schleyer` | one file per animal, plus per-dish metadata |
| `Berni`, `Arguello` | one file per animal, columns in a fixed order |
| `DeepLabCut` | DeepLabCut CSV/HDF5 exports, one file per video |

Two things follow from this :

- **If one of them matches your tracker**, this notebook works on your own data with only
  the first section changed.
- **If none does**, a new lab format can be described and registered, after which your data
  imports like any other.

A lab format carries nominal values for things like the frame rate and the arena, because
they are usually constant for a lab. They describe the lab, not any particular recording,
which is why the import prefers what it can measure in the data itself.

## References

> Greaney MR, Heckscher ES, Kaufman MT (2025) *Multiple scales of coordination along the
> body axis during Drosophila larval locomotion*. bioRxiv.
> <https://doi.org/10.1101/2025.08.21.671596>

> *DeepLabCut results*. figshare.
> <https://figshare.com/articles/dataset/DeepLabCut_results/31508029>

Please cite both if you use this data.